Cell 1 — Setup and feature function:

In [2]:
import pandas as pd
import numpy as np
import os
import logging
import gc
logging.getLogger('rdkit').setLevel(logging.CRITICAL)
os.environ['RDKIT_LOG_LEVEL'] = 'CRITICAL'

from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import AllChem

CSV_PATH = 'D:/DDI Project/data/drug_pairs_labeled.csv'
FEAT_DIR = 'D:/DDI Project/data/features/'
CHUNK    = 10_000
N_BITS   = 2048   # upgraded from 1024

os.makedirs(FEAT_DIR, exist_ok=True)

def get_fingerprint(smiles_string):
    mol = Chem.MolFromSmiles(str(smiles_string))
    if mol is None:
        return None
    return np.array(
        AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=N_BITS),
        dtype=np.float32
    )

def get_pair_features(smi_a, smi_b):
    """
    Builds an 8193-dimensional feature vector for a drug pair:

    Block 1 — fp_A (2048)       : Drug A fingerprint
    Block 2 — fp_B (2048)       : Drug B fingerprint
    Block 3 — fp_A AND fp_B (2048) : shared substructures
    Block 4 — |fp_A - fp_B| (2048) : structural difference
    Block 5 — cosine similarity (1) : overall structural similarity score

    Total: 4 x 2048 + 1 = 8193
    """
    fp_a = get_fingerprint(smi_a)
    fp_b = get_fingerprint(smi_b)

    if fp_a is None or fp_b is None:
        return None

    shared = fp_a * fp_b
    diff   = np.abs(fp_a - fp_b)

    # cosine similarity — single continuous score
    norm_a = np.linalg.norm(fp_a)
    norm_b = np.linalg.norm(fp_b)
    if norm_a == 0 or norm_b == 0:
        cosine = np.array([0.0], dtype=np.float32)
    else:
        cosine = np.array(
            [float(np.dot(fp_a, fp_b) / (norm_a * norm_b))],
            dtype=np.float32
        )

    return np.concatenate([fp_a, fp_b, shared, diff, cosine])

# Quick test
test = get_pair_features(
    'CC(=O)Oc1ccccc1C(=O)O',       # Aspirin
    'CC(C)Cc1ccc(cc1)C(C)C(=O)O'  # Ibuprofen
)
print(f"Feature vector shape : {test.shape}")    # should be (8193,)
print(f"Last value (cosine)  : {test[-1]:.4f}")  # cosine similarity
print(f"dtype                : {test.dtype}")
print("Cell 1 done")

Feature vector shape : (8193,)
Last value (cosine)  : 0.3266
dtype                : float32
Cell 1 done


Cell 2 — Quick 200-row validity check:

In [3]:
valid   = 0
invalid = 0

sample = pd.read_csv(CSV_PATH, nrows=200)

for _, row in sample.iterrows():
    feat = get_pair_features(row['Drug1_SMILES'], row['Drug2_SMILES'])
    if feat is not None:
        valid += 1
    else:
        invalid += 1

print(f"Valid   : {valid} / 200")
print(f"Invalid : {invalid} / 200")

# Check cosine similarity range across a sample
cosines = []
for _, row in sample.iterrows():
    feat = get_pair_features(row['Drug1_SMILES'], row['Drug2_SMILES'])
    if feat is not None:
        cosines.append(feat[-1])

cosines = np.array(cosines)
print(f"\nCosine similarity stats:")
print(f"  Min  : {cosines.min():.4f}")
print(f"  Max  : {cosines.max():.4f}")
print(f"  Mean : {cosines.mean():.4f}")
print(f"  Std  : {cosines.std():.4f}")

Valid   : 200 / 200
Invalid : 0 / 200

Cosine similarity stats:
  Min  : 0.0000
  Max  : 0.5334
  Mean : 0.1773
  Std  : 0.0820


Cell 3 — Count total rows and check disk space:

In [4]:
import shutil

TOTAL = len(pd.read_csv(CSV_PATH))
N_FEAT = 8193

free_gb   = shutil.disk_usage(FEAT_DIR).free / 1024**3
needed_gb = (TOTAL * N_FEAT * 4) / 1024**3   # float32 = 4 bytes

print(f"Total rows in dataset : {TOTAL:,}")
print(f"Feature vector size   : {N_FEAT}")
print(f"Space needed          : {needed_gb:.2f} GB")
print(f"Free disk space       : {free_gb:.1f} GB")

if free_gb < needed_gb + 2:
    print("WARNING: Low disk space. Free up space before Cell 4.")
else:
    print("Disk space OK.")

Total rows in dataset : 133,063
Feature vector size   : 8193
Space needed          : 4.06 GB
Free disk space       : 83.4 GB
Disk space OK.


Cell 4 — Build feature files using memory-mapped storage:

In [5]:
X_path = FEAT_DIR + 'X.npy'
y_path = FEAT_DIR + 'y.npy'

# Delete old files if they exist from a previous run
for path in [X_path, y_path]:
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted old {os.path.basename(path)}")

# Pre-allocate on disk — nothing in RAM yet
X_mm = np.lib.format.open_memmap(
    X_path, mode='w+', dtype=np.float32,
    shape=(TOTAL, N_FEAT)
)
y_mm = np.lib.format.open_memmap(
    y_path, mode='w+', dtype=np.uint8,
    shape=(TOTAL,)
)

write_pos = 0
chunk_num = 0
skipped   = 0

print(f"Building feature matrix for {TOTAL:,} pairs...")
print("This will take around 20-30 minutes. Do not close VS Code.\n")

for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK):
    chunk_num += 1

    for _, row in chunk.iterrows():
        feat = get_pair_features(row['Drug1_SMILES'], row['Drug2_SMILES'])
        if feat is not None:
            X_mm[write_pos] = feat
            y_mm[write_pos] = int(row['label'])
            write_pos += 1
        else:
            skipped += 1

    # Progress every 20 chunks = every 200,000 rows
    if chunk_num % 20 == 0:
        pct = (write_pos / TOTAL) * 100
        print(f"  Progress: {write_pos:,} / {TOTAL:,} rows ({pct:.1f}%)"
              f"  Skipped: {skipped}")
        gc.collect()

# Flush and close
del X_mm, y_mm
gc.collect()

print(f"\nDone.")
print(f"Rows written  : {write_pos:,}")
print(f"Rows skipped  : {skipped}")
print(f"Feature matrix: {os.path.getsize(X_path)/1024**3:.2f} GB")
print(f"Labels        : {os.path.getsize(y_path)/1024:.0f} KB")
print("Next: run 03_train_classical.ipynb")

Building feature matrix for 133,063 pairs...
This will take around 20-30 minutes. Do not close VS Code.


Done.
Rows written  : 133,063
Rows skipped  : 0
Feature matrix: 4.06 GB
Labels        : 130 KB
Next: run 03_train_classical.ipynb
